# 04 — Roster Composition → Win Probability

Builds a game-level dataset joining roster physical features with game outcomes,
then trains a Random Forest (and optionally a logistic regression) to predict
win probability.  Reports feature importances and cross-validated AUC.


In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.models import build_game_dataset, train_roster_model, plot_feature_importances

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

In [ ]:
master    = pd.read_csv('../data/processed/master_labeled.csv')
schedules = pd.read_csv('../data/raw/schedules/schedules_raw.csv')
print('Master:', master.shape, '| Schedules:', schedules.shape)

## Build game-level dataset

In [ ]:
game_df = build_game_dataset(master, schedules)
print('Game dataset shape:', game_df.shape)
print('Win rate:', game_df['win'].mean().round(3))
game_df.head()

## Train Random Forest

In [ ]:
if len(game_df) < 30:
    print('Not enough game rows yet — run notebook 01 first to collect data.')
else:
    rf_result = train_roster_model(game_df, model_type='rf')
    print(f"CV AUC: {rf_result['cv_auc_mean']:.3f} ± {rf_result['cv_auc_std']:.3f}")
    
    fig, ax = plt.subplots(figsize=(10, 7))
    plot_feature_importances(rf_result, top_n=15, ax=ax)
    plt.savefig('../data/processed/roster_rf_importances.png', dpi=150)
    plt.show()
    
    display(rf_result['importances'].head(15).rename('Importance').to_frame())

## Compare: Logistic Regression

In [ ]:
if len(game_df) >= 30:
    lr_result = train_roster_model(game_df, model_type='lr')
    print(f"Logistic CV AUC: {lr_result['cv_auc_mean']:.3f} ± {lr_result['cv_auc_std']:.3f}")
    display(lr_result['importances'].head(15).rename('|Coefficient|').to_frame())

## Roster matchup explorer — run-heavy vs pass-heavy opponents
Shows how roster composition interacts with opponent style.

In [ ]:
if len(game_df) >= 30 and 'opp_run_heavy' in game_df.columns and 'home_OL_avg_weight' in game_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for i, (label, mask) in enumerate([
        ('vs Run-Heavy Opponents', game_df['opp_run_heavy'] == 1),
        ('vs Pass-Heavy Opponents', game_df['opp_pass_heavy'] == 1)
    ]):
        sub = game_df[mask]
        if len(sub) < 10:
            axes[i].set_title(f'{label} (n too small)')
            continue
        axes[i].scatter(
            sub['home_OL_avg_weight'],
            sub['win'].astype(int) + np.random.normal(0, 0.03, len(sub)),
            alpha=0.4, c=sub['win'], cmap='RdYlGn', s=30
        )
        axes[i].set_xlabel('OL Avg Weight (lbs)')
        axes[i].set_ylabel('Win (jittered)')
        axes[i].set_title(label)
    plt.tight_layout()
    plt.savefig('../data/processed/roster_matchup_scatter.png', dpi=150)
    plt.show()
else:
    print('Insufficient data for matchup explorer — run notebook 01 first.')

## Interaction: OL weight × opponent run-heavy
Tests whether heavier OLs help more against run-heavy defenses.

In [ ]:
if len(game_df) >= 30 and 'opp_run_heavy' in game_df.columns and 'home_OL_avg_weight' in game_df.columns:
    import statsmodels.formula.api as smf
    df = game_df[['win','home_OL_avg_weight','opp_run_heavy']].dropna()
    df['OL_run_interaction'] = df['home_OL_avg_weight'] * df['opp_run_heavy']
    m = smf.logit('win ~ home_OL_avg_weight + opp_run_heavy + OL_run_interaction', data=df).fit()
    print(m.summary())
else:
    print('Insufficient data for interaction model.')